# 03 - WoE Scorecard (Logistic Regression)

## Purpose

This notebook covers the **Modelling** phase of the CRISP-DM and builds the project's primary model, the mentioned Weight-of-Evidence (WoE) logistic regression scorecard. The scorecard is the standard approach for retail PD models because every prediction can be traced back to a small number of binned features and their point contributions.

## Inputs

- `data/processed/train_prepared.parquet` and `test_prepared.parquet`, produced by notebook 02.

- `configs/preparation_params.json`, which lists the columns excluded from modelling (`SK_ID_CURR`, `TARGET`, `CODE_GENDER`).

## Approach

1. **WoE binning.** Each feature is divided into bins, and each bin is replaced by its Weight of Evidence. The binning trend per feature follow the findings of both notebooks 01 and 02 (monotonic, peak or unconstrained). Missing values receive their own bin.
2. **Feature Selection.** Features are ranked by Information Value. Weak features will be removed, and of each remaining highly correlated pair only the feature with the higher IV is kept.
3. **Logistic Regression.** on the WoE transformed features.
4. **Scorecard scaling.** Model coefficients are converted into points using a points-to-double-odds scale.
5. **Evaluation.** Discrimination with Gini, KS and calibration of the predicted probabilities.

## Evaluation rule

All fitting and all modelling choices are made on the **training set only**, using 5-fold cross-validation where a choice requires comparing alternatives. The **test set is used exactly once**, at the end of this notebook, to report the final performance. Using it earlier would turn it into part of the model development and bias the evaluation. 

## Output

the fitted scorecard. Its binning tables and its parameters, saved for the challenger comparison in Notebook 04 and the validation in Notebook 06. 


## Setup and data loading

The prepared training set and the parameters from notebook 02 are loaded. 
The columns listed in `NON_FEATURE_COLS` are separated from the features.
The identifier and the target are not predictors, and `CODE_GENDER` is excluded from modelling and retained only for fairness testing. 

The test set is deliberately **not loaded here**. It is loaded in the final evaluation section, evaluation rule is then enforceable in code rather than only a stated intention.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed = ROOT / "data" / "processed"
configs = ROOT / "configs"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Training data and parameters.

train_df = pd.read_parquet(processed / "train_prepared.parquet")
with open(configs / "preparation_params.json") as f:
    prep_params = json.load(f)

# Features separation.

NON_FEATURE_COLS = list(prep_params["non_feature_cols"])
feature_cols = [c for c in train_df.columns if c not in NON_FEATURE_COLS]

X_train = train_df[feature_cols]
y_train = train_df["TARGET"]

categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()
numerical_cols = X_train.select_dtypes(include="number").columns.tolist()

print(f"Training set: {X_train.shape[0]:,} rows, default rate {y_train.mean():.4f}")
print(f"Features: {len(feature_cols)} ({len(numerical_cols)} numerical, "
      f"{len(categorical_cols)} categorical)")
print(f"Excluded: {NON_FEATURE_COLS}")


Training set: 246,008 rows, default rate 0.0807
Features: 96 (81 numerical, 15 categorical)
Excluded: ['SK_ID_CURR', 'TARGET', 'CODE_GENDER']


## Weight of Evidence and Information Value

**Weight of Evidence (WoE)** measures how the risk of a bin relates to the risk of the whole population. It is the logarithm of a ratio of two shares: 

$$\text{WoE}_i = \ln\left(\frac{\%\text{Good}_i}{\%\text{Bad}_i}\right)$$

where %Good_i is the bin's share of **all** good borrowers and %Bad_i its share of **all** bad borrowers. Equivalently, the WoE of a bin is the difference between the bin's log-odds of being a good borrower and the population's log odds. A WoE of 0 means the bin has the population's average risk, a positive WoE means lower risk, and a negative WoE means higher risk. 

Example: 900 good and 100 bad borrowers (10% default rate), in three bins.

| Bin | Good | Bad | Default rate | %Good | %Bad | WoE |
|---|---|---|---|---|---|---|
| A | 270 | 30 | 10.0% | 0.300 | 0.30 | 0.000 |
| B | 380 | 20 | 5.0% | 0.422 | 0.20 | +0.747 |
| C | 250 | 50 | 16.7% | 0.278 | 0.50 | −0.588 |

Because WoE is on the log odds scale, it fits logistic regression, which models log odds as a linear function of inputs. Replacing each feature by the WoE of its bins also handles non-linear relationships, missing values (their own bin), outliers (absorbed into the outer bins) and categorical features in a single step.

**Information Value (IV)** summarises how well a whole feature separates good from bad borrowers:

$$\text{IV} = \sum_i (\%\text{Good}_i - \%\text{Bad}_i) \times \text{WoE}_i$$

Both factors always have the same sign, IV is never negative. In the example, IV = 0.297. Below 0.02 as not predictive, 0.02-0.1 as weak, 0.1-0.3 as medium and above 0.3 as strong, values above 0.5 should be checked for leakage. These thresholds are conventions from historical credit-scoring practice, not statistical tests. 

## Binning a single feature

Before binning all features, the procedure is examined in detail `EXT_SOURCE_2`: the strongest predictor in the EDA, with almost no missing values and a clean descending relationship with the default rate. The binning is constrained to be **descending**, meaning the default rate must fall from bin to bin as the score increases, consistent with notebook 01. 

The resulting binning table is used to verify three things: that the constraint holds, which sign convention the library uses for WoE, and the feature's total Information Value. 

In [2]:
from optbinning import OptimalBinning

feature = "EXT_SOURCE_2"
optb = OptimalBinning(
    name=feature,
    dtype="numerical",
    monotonic_trend="descending",
)
optb.fit(X_train[feature].values, y_train.values)

print(f"Status: {optb.status}")
binning_table = optb.binning_table.build()
binning_table

Status: OPTIMAL


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.14)",12734,0.051763,9926,2808,0.220512,-1.169797,0.114053,1.349559e-02
1,"[0.14, 0.26)",19146,0.077827,16461,2685,0.140238,-0.619169,0.038641,4.754411e-03
2,"[0.26, 0.32)",13381,0.054393,11844,1537,0.114864,-0.390493,0.009770,1.213516e-03
3,"[0.32, 0.38)",12649,0.051417,11243,1406,0.111155,-0.353485,0.007452,9.266288e-04
4,"[0.38, 0.45)",18812,0.076469,17050,1762,0.093664,-0.162781,0.002170,2.708935e-04
5,"[0.45, 0.49)",12388,0.050356,11338,1050,0.084759,-0.053112,0.000145,1.815416e-05
6,"[0.49, 0.56)",30605,0.124407,28247,2358,0.077046,0.050691,0.000313,3.911575e-05
7,"[0.56, 0.62)",32756,0.133150,30592,2164,0.066064,0.216298,0.005691,7.100078e-04
8,"[0.62, 0.64)",13077,0.053157,12335,742,0.056741,0.378365,0.006501,8.078387e-04
9,"[0.64, 0.66)",14273,0.058018,13534,739,0.051776,0.47518,0.010756,1.331973e-03


**What I see:** The binning status is `OPTIMAL`, `EXT_SOURCE_2` is divided into 14 bins, and the default rate decreases monotonically from 22.1% in the lowest score bin to 2.9% in the highest. The WoE rises accordingly, from -1.170 to +1.077. The total Information Value is 0.320. The `Missing` bin contains 533 applicants (0.2%) with a default rate of 8.3%, close to the overall rate of 8.1% and a WoE of approximately 0.

**Interpretation:** The descending constraint is satisfied. Recomputing the WoE of the riskiest bin by hand (ln(0.0439 / 0.1414) = - 1.170) confirms that optbinning follows the standard convention: negative WoE indicates higher than average risk, positive WoE lower than average risk. An IV of 0.320 classifies the feature as strong, and since it lies below 0.5 it gives no indication of leakage. More than half of the IV comes from the two outer bins (0.114 and 0.064), so most of the separation happens at the extremes of the score. Missingness of `EXT_SOURCE_2` carries almost no information, since its default rate is close to the population average, unlike the missingness of `EXT_SOURCE_1` found in notebook 01.

**Decisions:** The procedure works and is applied to all features. Fourteen bins are too many for a scorecard, since each bin becomes a separate line of points and narrow bins are less stable over time. A maximum number of bins is therefore set for the full binning accepting a small loss of IV in exchange for interpretability and stability. 

## Binning all features 

All 96 features are binned with optbinning's `BinningProcess`, which applies the same optimal binning process above to every feature. Three settings apply:

- **Trend.** The seven features whose shape was established in notebook 01 and 02 receives an explicit trend (descending, ascending, peak or unconstrained). All other features use automatic trend detection. 
- **At most 6 bins per feature**, a modelling choice that favours interpretability and stability over the last fraction of IV.
- **At least 5% of applicants per bin**, so that every WoE is estimated from a sufficient number of observations.

The binning is fitted on the training set only. The results are reviewed in a summary table, and individual binning tables are inspected only where a feature needs attention.

In [3]:
from optbinning import BinningProcess

KNOWN_TRENDS = {
    "EXT_SOURCE_1": "descending",
    "EXT_SOURCE_2": "descending",
    "EXT_SOURCE_3": "descending",
    "YEARS_BIRTH": "descending",
    "CREDIT_INCOME_RATIO": "peak",
    "ANNUITY_INCOME_RATIO": "ascending",
    "CREDIT_GOODS_RATIO": "ascending",
    "ANNUITY_CREDIT_RATIO": None,
}

missing = [f for f in KNOWN_TRENDS if f not in feature_cols]
assert not missing, f"Features not found: {missing}"

MAX_N_BINS = 6
MIN_BIN_SIZE = 0.05

binning_process = BinningProcess(
    variable_names=feature_cols,
    categorical_variables=categorical_cols,
    max_n_bins=MAX_N_BINS,
    min_bin_size=MIN_BIN_SIZE,
    binning_fit_params={f: {"monotonic_trend": t} for f, t in KNOWN_TRENDS.items()},
    n_jobs=1
)
binning_process.fit(X_train, y_train)
print("Binning complete.")


Binning complete.


In [4]:
summary = (
    binning_process.summary()
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

print("Status counts:", summary["status"].value_counts().to_dict())
print()
print(summary[["name", "dtype", "status", "n_bins", "iv"]].head(20).to_string())

# IV trade off of the 6 bin limit, compared with the previous 14 binning. 

iv_6 = summary.loc[summary["name"] == "EXT_SOURCE_2", "iv"].iloc[0]
print(f"\nEXT_SOURCE_2 IV: 14 bins = 0.320, {MAX_N_BINS} bins = {iv_6:.3f}")

Status counts: {'OPTIMAL': 96}

                           name        dtype   status n_bins        iv
0                  EXT_SOURCE_3    numerical  OPTIMAL      6  0.330948
1                  EXT_SOURCE_2    numerical  OPTIMAL      6   0.31191
2          ANNUITY_CREDIT_RATIO    numerical  OPTIMAL      6  0.167901
3                  EXT_SOURCE_1    numerical  OPTIMAL      5  0.140636
4                YEARS_EMPLOYED    numerical  OPTIMAL      6   0.11008
5                   YEARS_BIRTH    numerical  OPTIMAL      6  0.083033
6            CREDIT_GOODS_RATIO    numerical  OPTIMAL      6  0.077059
7               OCCUPATION_TYPE  categorical  OPTIMAL      6  0.076546
8             ORGANIZATION_TYPE  categorical  OPTIMAL      6  0.067553
9              NAME_INCOME_TYPE  categorical  OPTIMAL      4  0.057153
10                   AMT_CREDIT    numerical  OPTIMAL      6  0.055184
11  REGION_RATING_CLIENT_W_CITY    numerical  OPTIMAL      3  0.052666
12         REGION_RATING_CLIENT    numerical 

In [5]:
iv_bands = pd.cut(
    summary["iv"],
    bins=[0, 0.02, 0.1, 0.3, np.inf],
    labels=["not predictive (<0.02)", "weak (0.02-0.1)",
            "medium (0.1-0.3)", "strong (>=0.3)"],
    right=False 
)
print(iv_bands.value_counts().sort_index())

iv
not predictive (<0.02)    58
weak (0.02-0.1)           33
medium (0.1-0.3)           3
strong (>=0.3)             2
Name: count, dtype: int64


In [6]:
binning_process.get_binned_variable("CREDIT_INCOME_RATIO").binning_table.build()

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 1.00)",12848,0.052226,12031,817,0.063590,0.257121,0.003101,3.866002e-04
1,"[1.00, 1.53)",21645,0.087985,20065,1580,0.072996,0.10907,0.001000,1.249350e-04
2,"[1.53, 2.25)",37899,0.154056,34851,3048,0.080424,0.004114,0.000003,3.253943e-07
3,"[2.25, 4.85)",103426,0.420417,94189,9237,0.089310,-0.110396,0.005367,6.704931e-04
4,"[4.85, 7.13)",41543,0.168868,38367,3176,0.076451,0.059093,0.000575,7.189818e-05
5,"[7.13, inf)",28647,0.116447,26645,2002,0.069885,0.155973,0.002654,3.314073e-04
6,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000e+00
7,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000e+00
Totals,,246008,1.000000,226148,19860,0.080729,,0.012700,1.585659e-03


**What I see:**  All 96 features were binned with status `OPTIMAL`. Limiting the binning to 6 bins reduced the IV of `EXT_SOURCE_2` from 0.320 to 0.312, a loss of 2.5%. The IV is highly concentrated: 58 features are not predictive (IV < 0.02), 33 are weak, 3 are medium and only 2 are strong. The strongest features are `EXT_SOURCE_3` (0.331) and `EXT_SOURCE_2` (0.312), followed by the engineered `ANNUITY_CREDIT_RATIO` (0.168), which ranks above `EXT_SOURCE_1`. No feature exceeds an IV of 0.5. `CREDIT_INCOME_RATIO` follows the enforced peak shape, with the default rate rising from 6.4% to 8.9% and falling back to 7.0%, but its IV is only 0.013.

**Interpretation:** The 6 bin limit retains 97.5% of the information while more than halving the number of bins, so the trade-off favours interpretability and stability. Predictive power is concentrated in a few features, confirming that the model must combine many moderate signals. The strong position of `ANNUITY_CREDIT_RATIO` supports the notebook 02 finding that loan structure carries more signal than affordability. `CREDIT_INCOME_RATIO` shows that a stable pattern is not necessarily a useful one, its hump has been consistent since notebook 01, but the bins differ too little from the average risk to contribute meaningful information. The size of each bin reflects how the optimiser merged ranges of similar risk, not a lending preference. Since the data contain only granted loans, conclusions about lending policy cannot be drawn from it.

**Decision:** Features are selected in three stages: an IV filter removing features below 0.02, a correlation filter on the WoE transformed features keeping the higher-IV feature of each correlated pair, and a check that all logistic regression coefficients have the same sign. 

## Feature selection

Features are selected in three stages. The first two are applied here; the third requires a fitted model and follows in the logistic regression section.

1. **IV filter.** Features with an Information Value below 0.02 are removed as not predictive.
2. **Correlation filter.** The remaining features are transformed into their WoE values, which is the representation the model receives. For every pair with an absolute correlation above 0.7, only the feature with the higher IV is kept. Features are processed in descending order of IV, so a feature is only kept if it is not highly correlated with any stronger feature already kept. The threshold of 0.7 is a common convention in scorecard development and is documented as a modelling choice.
3. **Sign check** (Section 7). All coefficients of the logistic regression must have the same sign.

In the WoE transformation, missing values receive the WoE of their own `Missing` bin (`metric_missing="empirical"`) rather than the library default of 0, preserving the informative missingness established in notebook 02.

In [7]:
IV_THRESHOLD = 0.02

iv_by_feature = summary.set_index("name")["iv"].astype(float)
selected_iv = iv_by_feature[iv_by_feature >= IV_THRESHOLD].index.tolist()

print(f"Stage 1, IV >= {IV_THRESHOLD}: {len(selected_iv)} of {len(iv_by_feature)} features kept")
print("Removed with the highest IV:")
print(iv_by_feature[iv_by_feature < IV_THRESHOLD].sort_values(ascending=False).head(5).round(4).to_string())



Stage 1, IV >= 0.02: 38 of 96 features kept
Removed with the highest IV:
name
LANDAREA_AVG                  0.0199
FLOORSMIN_AVG                 0.0198
LIVINGAPARTMENTS_AVG          0.0190
YEARS_BUILD_AVG               0.0186
AMT_REQ_CREDIT_BUREAU_YEAR    0.0181


In [8]:
CORR_THRESHOLD = 0.7

# Woe transformation and missing values to their own bin.
X_train_woe = binning_process.transform(X_train, metric="woe", metric_missing="empirical")

#Processing of features from strongest to weakest.

ranked = iv_by_feature.loc[selected_iv].sort_values(ascending=False).index.tolist()
corr = X_train_woe[ranked].corr().abs()

kept, dropped = [], {}
for feat in ranked:
    stronger_partners = [k for k in kept if corr.loc[feat, k] > CORR_THRESHOLD]
    if stronger_partners:
        dropped[feat] = stronger_partners[0]
    else:
        kept.append(feat)

print(f"Stage 2 |corr| <= {CORR_THRESHOLD}: {len(kept)} feature kept, {len(dropped)} removed \n")
for feat, partner in dropped.items():
    print(f" removed {feat} (IV {iv_by_feature[feat]:.3f}) "
          f" -> correlated {corr.loc[feat, partner]:.2f} with {partner} (IV {iv_by_feature[partner]:.3f})")
selected_features = kept


Stage 2 |corr| <= 0.7: 24 feature kept, 14 removed 

 removed REGION_RATING_CLIENT (IV 0.050)  -> correlated 0.95 with REGION_RATING_CLIENT_W_CITY (IV 0.053)
 removed TOTALAREA_MODE (IV 0.039)  -> correlated 0.89 with FLOORSMAX_AVG (IV 0.041)
 removed LIVINGAREA_AVG (IV 0.037)  -> correlated 0.88 with FLOORSMAX_AVG (IV 0.041)
 removed ELEVATORS_AVG (IV 0.036)  -> correlated 0.91 with FLOORSMAX_AVG (IV 0.041)
 removed APARTMENTS_AVG (IV 0.035)  -> correlated 0.88 with FLOORSMAX_AVG (IV 0.041)
 removed YEARS_BEGINEXPLUATATION_AVG (IV 0.031)  -> correlated 0.83 with FLOORSMAX_AVG (IV 0.041)
 removed DAYS_EMPLOYED_ANOMALY (IV 0.031)  -> correlated 0.74 with NAME_INCOME_TYPE (IV 0.057)
 removed ENTRANCES_AVG (IV 0.029)  -> correlated 0.75 with FLOORSMAX_AVG (IV 0.041)
 removed WALLSMATERIAL_MODE (IV 0.027)  -> correlated 0.79 with FLOORSMAX_AVG (IV 0.041)
 removed HOUSING_MISSING_SHARE (IV 0.026)  -> correlated 0.76 with FLOORSMAX_AVG (IV 0.041)
 removed EMERGENCYSTATE_MODE (IV 0.025)  -> c

**What I see:** Stage 1 keeps 38 of 96 features. The closest exclusions lie just below the threshold (`LANDAREA_AVG` 0.0199, `FLOORSMIN_AVG` 0.0198). Stage 2 keeps 24 features and removes 14. `REGION_RATING_CLIENT` is removed in favour of `REGION_RATING_CLIENT_W_CITY` (correlation 0.95). Twelve housing related features, including `HOUSING_MISSING_SHARE`, are removed in favour of `FLOORSMAX_AVG`, the housing feature with the highest IV (0.041). `DAYS_EMPLOYED_ANOMALY` is removed in favour of `NAME_INCOME_TYPE` (correlation 0.74).

**Interpretation:** The high correlations within the housing block are driven by shared missingness as about 47% of applicants have no housing data, so they fall into the `Missing` bin of every housing feature at once. The signal that `HOUSING_MISSING_SHARE` was designed to preserve is therefore already carried by the `Missing` bin of `FLOORSMAX_AVG`. Similarly, `NAME_INCOME_TYPE` contains a `Pensioner` category, which captures most of the information of the employment anomaly flag. Four of the five closest stage 1 exclusions are housing features that stage 2 would also have removed, so the exact IV threshold does not affect the outcome for them.

**Decision:** The remaining 24 features are used to fit the logistic regression, where the sign check (stage 3) may reduce the number further.

## Logistic regression

A logistic regression is fitted on the 24 WoE transformed features. Under the convention for WoE, a higher WoE indicates a safer bin, so every coefficient should be **negative**. A positive coefficient indicates that a feature's effect is distorted by correlation with other features, such features are removed (stage 3 of feature selection). A coefficient close to -1 indicates that a feature contributes nearly independently, since -1 is the coefficient of a single WoE feature fitted on its own.

Performance is estimated with stratified 5 fold cross validation and reported as the Gini coefficient (2 x AUC - 1). Because the binning and the feature selection were fitted on the full training set, this estimate is slightly optimistic. The unbiased estimate will be obtained in section 9.

In [9]:
from sklearn.linear_model import LogisticRegression

x_train_set = X_train_woe[selected_features]

model = LogisticRegression(max_iter=1000)
model.fit(x_train_set, y_train)

coefs = pd.Series(model.coef_[0], index=selected_features).sort_values()
print(f"Intercept: {model.intercept_[0]:.4f}\n")
print(coefs.round(4).to_string())

# Coefficients that are positive dont follow the convention of the WoE we defined before. 

wrong_sign = coefs[coefs >= 0]
print(f"\nCoefficients with the wrong sign (>= 0): {len(wrong_sign)}")
if len(wrong_sign):
    print(wrong_sign.round(4).to_string())

Intercept: -2.4356

EXT_SOURCE_3                  -0.8215
EXT_SOURCE_2                  -0.7294
ANNUITY_CREDIT_RATIO          -0.5546
OWN_CAR_AGE                   -0.5499
EXT_SOURCE_1                  -0.5215
NAME_EDUCATION_TYPE           -0.4599
CREDIT_GOODS_RATIO            -0.4425
AMT_ANNUITY                   -0.3853
ORGANIZATION_TYPE             -0.3649
FLOORSMAX_AVG                 -0.3510
YEARS_EMPLOYED                -0.3450
NAME_FAMILY_STATUS            -0.2770
REG_CITY_NOT_LIVE_CITY        -0.2634
YEARS_ID_PUBLISH              -0.2606
OCCUPATION_TYPE               -0.2529
REGION_RATING_CLIENT_W_CITY   -0.2486
YEARS_REGISTRATION            -0.2384
YEARS_LAST_PHONE_CHANGE       -0.2157
FLAG_DOCUMENT_3               -0.2043
NAME_INCOME_TYPE              -0.0590
REGION_POPULATION_RELATIVE    -0.0230
AMT_CREDIT                     0.0203
REG_CITY_NOT_WORK_CITY         0.1136
YEARS_BIRTH                    0.2428

Coefficients with the wrong sign (>= 0): 3
AMT_CREDIT              

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(
    LogisticRegression(max_iter=1000), x_train_set, y_train,
    cv=cv, scoring="roc_auc"
)
gini_scores = 2 * auc_scores - 1

print("Gini per fold:", np.round(gini_scores, 4))
print(f"Mean Gini: {gini_scores.mean():.4f} (std {gini_scores.std():.4f})")

Gini per fold: [0.5023 0.4934 0.4883 0.5041 0.4916]
Mean Gini: 0.4959 (std 0.0062)


### Stage 3: sign check

Three features have positive coefficient, contradicting their own binning, a safer bin would raise the predicted risk. Since removing one feature changes all other coefficients, wrong signed features are removed one at a time, always the one with the largest positive coefficient and the model is refitted after each removal until every coefficient is negative. 

In [11]:
features = list(selected_features)
removed_stage3 = []

while True:
    m = LogisticRegression(max_iter=1000).fit(X_train_woe[features], y_train)
    c = pd.Series(m.coef_[0], index=features)
    if (c < 0).all():
        break
    worst = c.idxmax()
    removed_stage3.append((worst, round(float(c[worst]), 4)))
    features.remove(worst)

final_features = features
print(f"Removed in stage 3 (in order): {removed_stage3}")
print(f"Final number of features: {len(final_features)}")

Removed in stage 3 (in order): [('YEARS_BIRTH', 0.2428), ('REG_CITY_NOT_WORK_CITY', 0.1327), ('AMT_CREDIT', 0.0196)]
Final number of features: 21


In [12]:
final_model = LogisticRegression(max_iter=1000).fit(X_train_woe[final_features], y_train)
final_coefs = pd.Series(final_model.coef_[0], index=final_features).sort_values()

print(f"Intercept: {final_model.intercept_[0]:.4f}\n")
print(final_coefs.round(4).to_string())

auc_final = cross_val_score(
    LogisticRegression(max_iter=1000), X_train_woe[final_features], y_train,
    cv=cv, scoring="roc_auc",
)

gini_final = 2 * auc_final - 1
print(f"\n Mean Gini: {gini_final.mean():.4f} (std {gini_final.std():.4f})")
print(f"Before stage 3 (24 features): {gini_scores.mean():.4f}")

Intercept: -2.4356

EXT_SOURCE_3                  -0.8180
EXT_SOURCE_2                  -0.7218
OWN_CAR_AGE                   -0.5511
ANNUITY_CREDIT_RATIO          -0.5411
NAME_EDUCATION_TYPE           -0.4915
EXT_SOURCE_1                  -0.4838
CREDIT_GOODS_RATIO            -0.4452
AMT_ANNUITY                   -0.3872
FLOORSMAX_AVG                 -0.3343
ORGANIZATION_TYPE             -0.3266
YEARS_EMPLOYED                -0.2835
OCCUPATION_TYPE               -0.2600
REGION_RATING_CLIENT_W_CITY   -0.2488
NAME_FAMILY_STATUS            -0.2197
YEARS_LAST_PHONE_CHANGE       -0.2146
REG_CITY_NOT_LIVE_CITY        -0.1997
YEARS_ID_PUBLISH              -0.1949
FLAG_DOCUMENT_3               -0.1878
YEARS_REGISTRATION            -0.1560
REGION_POPULATION_RELATIVE    -0.0316
NAME_INCOME_TYPE              -0.0057

 Mean Gini: 0.4955 (std 0.0062)
Before stage 3 (24 features): 0.4959


In [13]:
# Optional check as the 2 features are almost close to 0.

near_zero = ["NAME_INCOME_TYPE", "REGION_POPULATION_RELATIVE"]
reduced_features = [f for f in final_features if f not in near_zero]

auc_reduced = cross_val_score(
    LogisticRegression(max_iter=1000), X_train_woe[reduced_features], y_train,
    cv=cv, scoring="roc_auc",
)

gini_reduced = 2 * auc_reduced - 1 
print(f"{len(reduced_features)} features: mean Gini {gini_reduced.mean():.4f}"
      f" (std {gini_reduced.std():.4f})")
print(f"{len(final_features)} features: mean Gini {gini_final.mean():.4f}")

19 features: mean Gini 0.4955 (std 0.0062)
21 features: mean Gini 0.4955


In [14]:
# Decision: drop the two near zero features.
final_features = reduced_features
final_model = LogisticRegression(max_iter=1000).fit(X_train_woe[final_features], y_train)
coefs = pd.Series(final_model.coef_[0], index=final_features)

print(f"Final scorecard: {len(final_features)} characteristics")
assert (coefs < 0).all(), "A coefficient has the wrong sign"

Final scorecard: 19 characteristics


**What I see:** The logistic regression on 24 WoE features has an intercept of -2.436, matching the population log-odds of default (ln(0.0807/ 0.9193) = -2.433). Three coefficients had the wrong sign: `YEARS_BIRTH` (+0.243), `REG_CITY_NOT_WORK_CITY` (+0.133 after the first refit) and `AMT_CREDIT` (+0.020). They were removed one at a time, and removing `YEARS_BIRTH` did not correct the other two. The resulting all 21 features all have negative coefficients. The cross-validated Gini is 0.4955 (std 0.0062), against 0.4959 before stage 3. Two features have coefficients close to zero: `NAME_INCOME_TYPE` and `REGION_POPULATION_RELATIVE`. Removing them doesn't change the Gini.

**Interpretation:** The three removed features carried no information beyond the remaining features,the Gini change of 0.0004 is far smaller than the variation between folds. Age predicts default on its own (notebook 01), but its information is covered by other features in the combined model too, its removal also reduces the model's direct dependence on a sensitive attribute, although age remains part of the fairness analysis planned for notebook 06, since other features may act as proxies for it. The external scores contribute almost independently (coefficients close to −1). The low standard deviation across folds indicates stable performance. The cross-validated Gini is slightly optimistic because binning and feature selection were fitted on the full training set, indicating a small leakage.

**Decision:** The two near zero features are removed for parsimony, giving a final scorecard of 19 characteristics.

## Scorecard scaling

The model's log-odds are now converted into scorecard outputs. Each bin of each characteristic receives a fixed number of points of their bins, higher scores indicate lower risk. The scale is defined by a target score of 600 at good:bad odds of 50:1, with 20 points to double the odds (PDO - Points to double the odds). These parameters are a presentation and literature convention and do not affect how the model ranks applicants. 

After some research, following Siddiqi, the scaling constants are:


$$
\text{Factor} = \frac{\text{PDO}}{\ln 2},
\qquad
\text{Offset} = 600 - \text{Factor} \cdot \ln 50,
$$

and an applicant's score is

$$
\text{Score} = \text{Offset} + \text{Factor} \cdot \ln(\text{odds of good}).
$$

The points of bin $i$ of characteristic $j$ are

$$
\text{Points}_{ij} = -\left( \beta_j \cdot \text{WoE}_{ij} + \frac{\beta_0}{n} \right) \cdot \text{Factor} + \frac{\text{Offset}}{n},
$$

where $\beta_0$ is the intercept, $\beta_j$ the coefficient of characteristic
$j$ and $n$ the number of characteristics, so the points of an applicant's
bins sum to their total score.

In [15]:
PDO = 20
TARGET_SCORE = 600
TARGET_ODDS = 50

# Following through with the definition of the formula according to Siddiqi

factor = PDO / np.log(2)
offset = TARGET_SCORE - factor * np.log(TARGET_ODDS)
beta0 = final_model.intercept_[0]
betas = pd.Series(final_model.coef_[0], index=final_features)
n = len(final_features)

rows = []
for feat in final_features:
    table = binning_process.get_binned_variable(feat).binning_table.build().iloc[:-1]
    woe = table["WoE"].astype(float)
    points = -(betas[feat] * woe + beta0 / n) * factor + offset / n
    rows.append(pd.DataFrame({
        "characteristic": feat,
        "bin": table["Bin"].astype(str),
        "count": table["Count"],
        "event_rate": table["Event rate"].astype(float).round(4),
        "woe": woe.round(4),
        "points": points.round().astype(int),
    }))

scorecard = pd.concat(rows, ignore_index=True)
print(f"Factor: {factor:.4f}, Offset: {offset:.4f}")
scorecard[scorecard["characteristic"] == "EXT_SOURCE_2"]

Factor: 28.8539, Offset: 487.1229


,characteristic,bin,count,event_rate,woe,points
8,EXT_SOURCE_2,"(-inf, 0.14)",12734,0.2205,-1.1698,5
9,EXT_SOURCE_2,"[0.14, 0.26)",19146,0.1402,-0.6192,17
10,EXT_SOURCE_2,"[0.26, 0.45)",44842,0.1049,-0.2888,23
11,EXT_SOURCE_2,"[0.45, 0.62)",75749,0.0736,0.1008,31
12,EXT_SOURCE_2,"[0.62, 0.71)",59392,0.0501,0.5104,40
13,EXT_SOURCE_2,"[0.71, inf)",33612,0.0319,0.9805,50
14,EXT_SOURCE_2,Special,0,0.0000,0.0000,29
15,EXT_SOURCE_2,Missing,533,0.0826,-0.0243,29


In [16]:
# Within every characteristic, a higher WoE (safer bin) must earn more points.
def points_follow_woe(t):
    return t.loc[t["woe"].idxmax(), "points"] >= t.loc[t["woe"].idxmin(), "points"]

consistent = scorecard.groupby("characteristic").apply(points_follow_woe)
assert consistent.all(), f"Points reversed for: {list(consistent[~consistent].index)}"
print("Points increase with WoE in all characteristics.")

Points increase with WoE in all characteristics.


In [17]:
# Computing the scores, @ stands for matrix multiplication.

log_odds_default = beta0 + X_train_woe[final_features] @ betas.values
score_formula = offset + factor * (-log_odds_default) 

# Must equal now the score implied by the model's predicted proababilities.

p_default = final_model.predict_proba(X_train_woe[final_features])[:, 1]
score_model = offset + factor * np.log((1 - p_default) / p_default)

assert np.allclose(score_formula, score_model), "Scaling does not match the model"
print("Scaling verified against the model's probabilities.")
print(score_formula.describe().round(1))


Scaling verified against the model's probabilities.
count    246008.0
mean        567.8
std          27.6
min         460.1
25%         549.1
50%         568.4
75%         587.1
max         668.0
dtype: float64


**What I see:** The scaling constants are Factor = 28.85 and Offset = 487.12. For `EXT_SOURCE_2`, points range from 5 in the riskiest bin (default rate 22.1%) to 50 in the safest bin (3.2%). Across the training set, scores range from 460 to 668, with a median of 568 and a standard deviation of 27.6. The scores computed from the scaling formula match the model's predicted probabilities exactly, and within every characteristic, bins with a higher WoE receive more points.

**Interpretation:** The difference of about 45 points between the riskiest and safest `EXT_SOURCE_2` bins corresponds to 45 / 20 = 2.25 doublings of the odds, so, all else equal, the safest bin has about 4.8 times better odds than the riskiest. The median score of 568 corresponds to odds of about 16.5:1, a default probability of about 5.7%, below the average rate of 8.1%: defaults are concentrated in a minority of low-scoring applicants. During development, a stale cell output displayed reversed points, the check that points increase with WoE was added to verify the delivered table directly, rather than only the scaling formula.

**Decision:** The scorecard with 19 characteristics is final. Its performance is now evaluated once, on the test set.

## Final evaluation on the test set

The test set is used here for the first and only time. It is transformed with the binning fitted on the training set, including the WoE of each `Missing` bin, and scored with the final 19-characteristic model. The model is not changed after this evaluation.

Three aspects are evaluated:

- **Discrimination (Gini).** Compared with the cross-validated Gini of 0.4955 to assess how much the development estimate was optimistic.
- **Discrimination (KS).** The maximum distance between the cumulative score distribution of defaulters and non-defaulters. Used for comparison in quantiles between the true- and false positives.

$$
\text{KS} = \max_s \left| F_{\text{bad}}(s) - F_{\text{good}}(s) \right|
$$

- **Calibration.** Whether predicted default probabilities match observed
  default rates, assessed by decile of predicted PD and by the Brier score,

$$
\text{Brier} = \frac{1}{N}\sum_{i=1}^{N}(p_i - y_i)^2,
$$

Compared with the baseline of predicting the average default rate for every applicant. Calibration matters for the IFRS 9 appendix, where the PD will be used for the expected credit loss directly. 

In [18]:
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss

# use of the test set.
test_df = pd.read_parquet(processed / "test_prepared.parquet")
X_test = test_df[feature_cols]
y_test = test_df["TARGET"]

# Same transformation as was used for the training set.
X_test_woe = binning_process.transform(X_test, metric="woe", metric_missing="empirical")
assert not X_test_woe[final_features].isnull().any().any(), "Untransformed values in test"

p_test = final_model.predict_proba(X_test_woe[final_features])[:, 1]
score_test = offset + factor * np.log((1 - p_test) / p_test)

print(f"Test set: {len(y_test):,} applicants, default rate {y_test.mean():.4f}")

Test set: 61,503 applicants, default rate 0.0807


In [19]:
auc_test = roc_auc_score(y_test, p_test)
gini_test = 2 * auc_test - 1

# False positive and True positive rate

fpr, tpr, _ = roc_curve(y_test, p_test)
ks_test = np.max(tpr - fpr)

brier_test = brier_score_loss(y_test, p_test)
brier_baseline = y_test.mean() * (1 - y_test.mean())

print(f"Gini: test {gini_test:.4f} vs cross-validated {gini_reduced.mean():.4f}")
print(f"ks: {ks_test:.4f}")
print(f"Brier: {brier_test:.4f} vs baseline {brier_baseline:.4f}") 



Gini: test 0.5001 vs cross-validated 0.4955
ks: 0.3695
Brier: 0.0685 vs baseline 0.0742


In [20]:
calib = pd.DataFrame({"predicted_pd": p_test, "default": y_test.values})
calib["decile"] = pd.qcut(calib["predicted_pd"], q=10, labels=False) + 1

calibration_table = calib.groupby("decile").agg(
    n=("default", "size"),
    mean_predicted=("predicted_pd", "mean"), 
    observed_rate=("default", "mean")
).round(4)

print(f"Mean predicted PD: {p_test.mean():.4f} vs observed default rate: {y_test.mean():.4f}\n")
calibration_table


Mean predicted PD: 0.0807 vs observed default rate: 0.0807



,n,mean_predicted,observed_rate
decile,,,
1,6151,0.0125,0.0150
2,6150,0.0218,0.0195
3,6150,0.0302,0.0272
4,6150,0.0392,0.0382
5,6151,0.0498,0.0478
6,6150,0.0630,0.0678
7,6150,0.0803,0.0797
8,6150,0.1044,0.1029
9,6150,0.1448,0.1502


**What I see:** On the 61,503 test applicants, the scorecard achieves a Gini of 0.5001, against a cross validated Gini of 0.4955 during development, and a KS of 0.370. The observed default rate rises monotonically across the deciles of predicted PD, from 1.5% to 25.9%. The riskiest 20% of applicants account for about half of all defaults. The Brier score is 0.0685, against 0.0742 for the baseline of predicting the average default rate. Within each decile, predicted and observed default rates differ by at most 0.54 percentage points, for both directions. 

**Interpretation:** The test Gini lies within the variation between the cross validation folds (std of 0.0062), so the model holds well to unseen applicants without measurable loss. The optimism expected from the binning and feature selection on the full training set is not meaningful here and negligible in practice. The model is also well calibrated, the deciles gaps show no systematic direction and lie within about 1.6 standard errors of the observed rate, so the predicted PDs can be used as probabilities, which will be used for the Expected Loss. The modest Brier Improvement reflects the low default rate rather than weak performance. 

**Decision:** The 19 characteristic scorecard is accepted as the primary model. Its test performance (Gini 0.500, KS 0.370) is the benchmark for the to be done LightGBM model in notebook 04. A formal calibration test is deferred to the validation notebook. 

## Saving the scorecard

The outputs are saved in three groups. The scorecard table, the calibration table and the test set metrics are written to `reports/`, as the human readable results and the benchmark for the challenger in notebook 04. The selected features and all modelling decisions as binning settings, selection thresholds, scaling parameters and coefficients are written to `configs/scorecard_params.json`, so that the model can be reproduced. The fitted binning and logistic regression are saved to `models/` for scoring later on. These files are excluded from the version control, since running this notebook simply regenerates them. 

In [21]:
import joblib

reports = ROOT / "reports"
models = ROOT / "models"
models.mkdir(exist_ok=True)

# Human readable results 

scorecard.to_csv(reports / "scorecard_table.csv", index=False)
calibration_table.to_csv(reports / "scorecard_calibration_test.csv")

# Benchmark

metrics = {
    "n_features": len(final_features),
    "cv_gini_mean": float(gini_reduced.mean()),
    "cv_gini_std": float(gini_reduced.std()),
    "test_gini": float(gini_test),
    "test_ks": float(ks_test),
    "test_brier": float(brier_test),
    "brier_baseline": float(brier_baseline),
}
with open(reports / "scorecard_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)


# Modelling decisions for reproducibility
   
scorecard_params = {
    "features": final_features,
    "binning": {
        "max_n_bins": MAX_N_BINS,
        "min_bin_size": MIN_BIN_SIZE,
        "known_trends": KNOWN_TRENDS,
    },
    "selection": {
        "iv_threshold": IV_THRESHOLD,
        "corr_threshold": CORR_THRESHOLD,
        "removed_wrong_sign": [name for name, _ in removed_stage3],
        "removed_near_zero": near_zero,
    },
    "scaling": {
        "pdo": PDO,
        "target_score": TARGET_SCORE,
        "target_odds": TARGET_ODDS,
        "factor": float(factor),
        "offset": float(offset),
    },
    "model": {
        "intercept": float(beta0),
        "coefficients": {f: float(b) for f, b in betas.items()},
    },
}
with open(configs / "scorecard_params.json", "w") as f:
    json.dump(scorecard_params, f, indent=2)

# Fitted objects for scoring

joblib.dump(binning_process, models / "binning_process.joblib")
joblib.dump(final_model, models / "scorecard_model.joblib")

# Reload check with the same test_predictions results

reloaded = joblib.load(models / "scorecard_model.joblib")
assert np.allclose(reloaded.predict_proba(X_test_woe[final_features])[:, 1], p_test)

print("Saved: scorecard table, calibration table, metrics, parameters, binning and model.")

Saved: scorecard table, calibration table, metrics, parameters, binning and model.


## Summary and handoff to notebook 04

This notebook developed a with 19 characteristics WoE logistic-regression scorecard. All 96 features were binned with at most 6 bins and at least 5% of applicants per bin, using the explicit trends where the shape was known previously. The features were selected in 3 stages (IV >= 0.02, WoE correlation <= 0.7, negative coefficients) and two features with near zero coefficients were removed. The model was scaled to 600 points of 50:1, with 20 points to double the odds.

**Results on the held out test set:** Gini 0.500 (cross-validated 0.496), KS 0.370, Brier score 0.0685 against a baseline of 0.0742, with predicted and observed default rates agreeing within 0.54 percentage points in every decile. 

**Notebook 04 receives:**
- The same prepared train and test sets, and the same evaluation rule.
- `reports/scorecard_metrics.json`, the benchmark the LightGBM is compared against.

**Open points:**
- A formal calibration test and stability analysis (notebook 06).
- Fairness evaluation by gender and age, including proxy effects, since both attributes are excluded from the model but may be reflected in other features (notebook 06).
- The data contain only granted loans; the effect of this selection bias is discussed in the validation notebook.